In [28]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM


In [29]:
#Loading feature vectors
vision_features = torch.load('../Encoder/features.pt')
labels = torch.load('../Encoder/labels.pt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vision_features = vision_features.to(device)

In [30]:
#Text tokenizer and Encoder
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")
text_encoder = AutoModel.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract").to(device)

#Freeze text encoder parameters
for param in text_encoder.parameters():
    param.requires_grad = False

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7272.13it/s]
BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
#LLM report generation

llm = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

for p in llm.parameters():
    p.requires_grad = False

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2524.31it/s]


In [32]:
#Diemensions
vision_dim = vision_features.shape[1]
text_dim = text_encoder.config.hidden_size
hidden_dim = 512
llm_dim = llm.config.n_embd

In [33]:
#Projection layer
image_projection = nn.Linear(vision_dim, hidden_dim).to(device)
text_projection = nn.Linear(text_dim, hidden_dim).to(device)
llm_projection = nn.Linear(hidden_dim, llm_dim).to(device)

In [34]:
#Text encoder
def encode_text(text_list):
    inputs = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = text_encoder(**inputs)
    
    #CLS token representation, summary of sentence
    return outputs.last_hidden_state[:, 0, :]

In [35]:
#Fusion module
def fuse(vision_emb, text_emb):
    return vision_emb + text_emb

In [36]:
#Full forward pass

def forward(vision_feats, text_inputs):

    #Text
    text_emb = encode_text(text_inputs)
    text_emb = text_projection(text_emb)

    #Vision
    vision_emb = image_projection(vision_feats)

    #Fusion
    fused_emb = fuse(vision_emb, text_emb)

    #Project to LLM space
    llm_input = llm_projection(fused_emb)

    #gpt expects sequence
    llm_input = llm_input.unsqueeze(1)

    #Generate report
    outputs = llm(inputs_embeds=llm_input)

    return outputs.logits

In [37]:

# =========================
# 6. CLINICAL PROMPT (REPORT STYLE)
# =========================
def build_prompt():
    return (
        "Generate a radiology report for a CT abdomen scan. "
        "Describe liver, kidneys, bowel, and any abnormalities in full sentences."
    )

In [ ]:
# =========================
# 11. OPTIONAL: REAL TEXT GENERATION
# =========================
def generate_text(vision_feats, max_len=60):

    prompt = build_prompt()

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        text_out = text_encoder(**inputs)

    text_emb = text_projection(text_out.last_hidden_state[:, 0, :])

    vision_emb = image_projection(vision_feats)

    fused = fuse(vision_emb, text_emb)

    generated = llm_projection(fused).unsqueeze(1)  # [B,1,768]

    output_tokens = []

    for _ in range(max_len):

        out = llm(inputs_embeds=generated)
        next_token_logits = out.logits[:, -1, :]

        next_token = torch.argmax(next_token_logits, dim=-1)

        output_tokens.append(next_token)

        next_emb = llm.transformer.wte(next_token).unsqueeze(1)

        generated = torch.cat([generated, next_emb], dim=1)

    # decode final sequence
    output_tokens = torch.stack(output_tokens, dim=1)

    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

In [ ]:
# =========================
# 12. RUN
# =========================
if __name__ == "__main__":

    print("\nGenerated Report:")
    print(generate_text(vision_features[:1]))

Image 1: The CT abdomen appears normal. No signs of hepatic steatosis or acute abdominal abnormality are detected.
Image 2: The CT abdomen appears normal. No signs of hepatic steatosis or acute abdominal abnormality are detected.
